# DJIWaypointTool

This notebook provides an interactive ipyleaflet map for:

- Planning a lawnmower flight path (waypoints) around a draggable mission-center pin
- Importing and visualizing multiple KML or KMZ trail files, each as an independent layer with its own color

Run the notebook top to bottom, but the final cell is designed to be safe to re-run.

In [ ]:
import sys
print(sys.version)


In [ ]:
import ipyleaflet, ipywidgets, shapely, pyproj, lxml
print('ipyleaflet', ipyleaflet.__version__)
print('ipywidgets', ipywidgets.__version__)
print('shapely', shapely.__version__)
print('pyproj', pyproj.__version__)
print('lxml', lxml.__version__)


In [ ]:
# Cell: Integrated UI (single source of truth for map, waypoint planner, and trail layers)
#
# Goals:
#   - Keep the full lawnmower waypoint generator (not a stub)
#   - Provide a waypoint panel (mission parameters, regenerate, clear)
#   - Provide a trail import panel supporting multiple KML/KMZ uploads
#   - Avoid duplicate "Show Panel" buttons when re-running the cell
#
# Notes:
#   - This cell intentionally closes any previous ipyleaflet Map widget named `m`
#   - All state is rebuilt deterministically each run, so the notebook is re-runnable

from IPython.display import clear_output
clear_output(wait=True)

import io
import zipfile
import math

import ipywidgets as widgets
from ipywidgets import Layout
from lxml import etree
from shapely.geometry import LineString
from pyproj import CRS, Transformer

from ipyleaflet import (
    Map, Marker, basemaps, basemap_to_tiles, SearchControl, WidgetControl,
    LayerGroup, Polyline, CircleMarker, Popup
)

# -----------------------------
# Tear down any previously rendered map to prevent orphan WidgetControls
# -----------------------------
try:
    if "m" in globals() and hasattr(m, "close"):
        m.close()
except Exception:
    pass

# -----------------------------
# Projection helpers
# -----------------------------
def utm_crs_for_latlon(lat: float, lon: float) -> CRS:
    zone = int(math.floor((lon + 180) / 6) + 1)
    is_northern = lat >= 0
    return CRS.from_dict({"proj": "utm", "zone": zone, "south": not is_northern})

def make_transformers(lat: float, lon: float):
    utm = utm_crs_for_latlon(lat, lon)
    wgs84 = CRS.from_epsg(4326)
    to_utm = Transformer.from_crs(wgs84, utm, always_xy=True)
    to_wgs84 = Transformer.from_crs(utm, wgs84, always_xy=True)
    return utm, to_utm, to_wgs84

# -----------------------------
# Lawnmower waypoint generator (meter-accurate)
# -----------------------------
def lawnmower_grid(center_lat, center_lon, width_m=200, height_m=120, spacing_m=20, bearing_deg=0):
    _, to_utm, to_wgs84 = make_transformers(center_lat, center_lon)

    cx, cy = to_utm.transform(center_lon, center_lat)

    half_w = width_m / 2
    half_h = height_m / 2

    lines = []
    y = cy - half_h
    direction = 1

    while y <= cy + half_h + 1e-6:
        if direction == 1:
            p1 = (cx - half_w, y)
            p2 = (cx + half_w, y)
        else:
            p1 = (cx + half_w, y)
            p2 = (cx - half_w, y)

        lines.append(LineString([p1, p2]))
        y += spacing_m
        direction *= -1

    # Rotation about center
    theta = math.radians(bearing_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    def rotate(pt):
        x, y = pt
        dx, dy = x - cx, y - cy
        rx = cx + dx * cos_t - dy * sin_t
        ry = cy + dx * sin_t + dy * cos_t
        return (rx, ry)

    waypoints_utm = []
    for ln in lines:
        for pt in ln.coords:
            waypoints_utm.append(rotate(pt))

    waypoints_ll = []
    for x, y in waypoints_utm:
        lon, lat = to_wgs84.transform(x, y)
        waypoints_ll.append((lat, lon))

    return waypoints_ll

# -----------------------------
# Map setup
# -----------------------------
default_center = (8.613934, -77.858251)  # River anchor from your test data
m = Map(
    center=default_center,
    zoom=14,
    scroll_wheel_zoom=True,
    basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik),
    layout=Layout(width="100%", height="800px")
)

search = SearchControl(
    position="topleft",
    url="https://nominatim.openstreetmap.org/search?format=json&q={s}",
    zoom=16,
    marker=None
)
m.add_control(search)

# Click-to-move mission center pin
pin = Marker(location=default_center, draggable=True)
m.add(pin)

# -----------------------------
# Overlays
# -----------------------------
flight_overlay = LayerGroup()
m.add_layer(flight_overlay)

trail_overlay_root = LayerGroup()
m.add_layer(trail_overlay_root)

# -----------------------------
# Waypoint planner state and rendering
# -----------------------------
launch_marker = None
waypoint_markers = []
path_polyline = None

def _clear_flight_overlay():
    global launch_marker, waypoint_markers, path_polyline

    if launch_marker is not None:
        try:
            flight_overlay.remove_layer(launch_marker)
        except Exception:
            pass
        launch_marker = None

    for mk in list(waypoint_markers):
        try:
            flight_overlay.remove_layer(mk)
        except Exception:
            pass
    waypoint_markers.clear()

    if path_polyline is not None:
        try:
            flight_overlay.remove_layer(path_polyline)
        except Exception:
            pass
        path_polyline = None

def _render_flight_path(waypoints, color="#FF0000"):
    global path_polyline, waypoint_markers, launch_marker

    _clear_flight_overlay()

    # Launch marker is the first waypoint for now
    if waypoints:
        launch_marker = CircleMarker(location=waypoints[0], radius=8, color=color, fill_color=color)
        flight_overlay.add_layer(launch_marker)

    # Polyline path
    path_polyline = Polyline(locations=waypoints, weight=4, color=color)
    flight_overlay.add_layer(path_polyline)

    # Optional markers at each waypoint, can be heavy for dense spacing
    if len(waypoints) <= 200:
        for i, (lat, lon) in enumerate(waypoints):
            mk = CircleMarker(location=(lat, lon), radius=3, color=color, fill_color=color)
            waypoint_markers.append(mk)
            flight_overlay.add_layer(mk)

# -----------------------------
# Waypoint panel UI
# -----------------------------
wp_status = widgets.HTML(value="<b>Flight Path:</b> not generated")

mission_name = widgets.Text(value="Test_Lawnmower", description="Name", layout=Layout(width="340px"))
altitude_m = widgets.IntSlider(value=70, min=10, max=300, step=1, description="Alt m", layout=Layout(width="340px"))
speed_mps = widgets.FloatSlider(value=6.0, min=1.0, max=15.0, step=0.5, description="Speed", layout=Layout(width="340px"))

width_m = widgets.IntSlider(value=250, min=20, max=2000, step=10, description="Width m", layout=Layout(width="340px"))
height_m = widgets.IntSlider(value=150, min=20, max=2000, step=10, description="Height m", layout=Layout(width="340px"))
spacing_m = widgets.IntSlider(value=25, min=5, max=200, step=1, description="Spacing", layout=Layout(width="340px"))
bearing_deg = widgets.IntSlider(value=30, min=0, max=359, step=1, description="Bearing", layout=Layout(width="340px"))

regen_btn = widgets.Button(description="Generate Flight Path", layout=Layout(width="170px"))
clear_btn = widgets.Button(description="Clear Flight Path", layout=Layout(width="170px"))

toggle_wp_panel_btn = widgets.ToggleButton(
    value=True,
    description="Hide Waypoint Panel",
    tooltip="Hide or show waypoint planner controls"
)

wp_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Waypoint Planner</b>"),
        widgets.HTML("Drag the pin or click the map to reposition the mission center."),
        mission_name,
        altitude_m,
        speed_mps,
        width_m,
        height_m,
        spacing_m,
        bearing_deg,
        widgets.HBox([regen_btn, clear_btn]),
        wp_status,
    ],
    layout=Layout(width="360px")
)

wp_panel_container = widgets.VBox(
    [widgets.HBox([toggle_wp_panel_btn]), wp_controls_box],
    layout=Layout(width="380px")
)

def _update_wp_panel_visibility(change=None):
    if toggle_wp_panel_btn.value:
        toggle_wp_panel_btn.description = "Hide Waypoint Panel"
        wp_controls_box.layout.display = "flex"
        wp_panel_container.layout.width = "380px"
    else:
        toggle_wp_panel_btn.description = "Show Waypoint Panel"
        wp_controls_box.layout.display = "none"
        wp_panel_container.layout.width = "180px"

toggle_wp_panel_btn.observe(_update_wp_panel_visibility, names=["value"])
_update_wp_panel_visibility()

wp_panel_control = WidgetControl(widget=wp_panel_container, position="topright")
m.add_control(wp_panel_control)

def _generate_waypoints_from_pin():
    lat, lon = pin.location
    wps = lawnmower_grid(
        center_lat=lat,
        center_lon=lon,
        width_m=width_m.value,
        height_m=height_m.value,
        spacing_m=spacing_m.value,
        bearing_deg=bearing_deg.value,
    )
    return wps

def _regen_clicked(b=None):
    wps = _generate_waypoints_from_pin()
    _render_flight_path(wps, color="#FF0000")
    lat, lon = pin.location
    wp_status.value = (
        f"<b>Flight Path:</b> {len(wps)} waypoint(s), "
        f"center {lat:.6f}, {lon:.6f}, alt {altitude_m.value} m, speed {speed_mps.value} m/s"
    )

def _clear_clicked(b=None):
    _clear_flight_overlay()
    wp_status.value = "<b>Flight Path:</b> cleared"

regen_btn.on_click(_regen_clicked)
clear_btn.on_click(_clear_clicked)

# Map click moves the pin, so regen becomes a one click operation
def _on_map_click(**kwargs):
    if kwargs.get("type") == "click":
        latlon = kwargs.get("coordinates")
        if latlon:
            pin.location = tuple(latlon)

m.on_interaction(_on_map_click)

# -----------------------------
# Trail import, multi-layer with per-layer color
# -----------------------------
trail_upload = widgets.FileUpload(accept=".kml,.kmz", multiple=True, description="Upload KML/KMZ")
trail_clear_btn = widgets.Button(description="Clear All Trails", layout=Layout(width="170px"))
trail_status = widgets.HTML(value="<b>Trails:</b> none loaded")
layers_box = widgets.VBox([], layout=Layout(width="360px"))

toggle_trail_panel_btn = widgets.ToggleButton(
    value=True,
    description="Hide Trail Panel",
    tooltip="Hide or show trail import controls"
)

trail_controls_box = widgets.VBox(
    [
        widgets.HTML("<b>Trail Import</b>"),
        widgets.HTML("Each upload becomes its own layer, with an independent color."),
        trail_upload,
        trail_clear_btn,
        trail_status,
        widgets.HTML("<b>Loaded Trail Layers</b>"),
        layers_box,
    ],
    layout=Layout(width="380px")
)

trail_panel_container = widgets.VBox(
    [widgets.HBox([toggle_trail_panel_btn]), trail_controls_box],
    layout=Layout(width="400px")
)

def _update_trail_panel_visibility(change=None):
    if toggle_trail_panel_btn.value:
        toggle_trail_panel_btn.description = "Hide Trail Panel"
        trail_controls_box.layout.display = "flex"
        trail_panel_container.layout.width = "400px"
    else:
        toggle_trail_panel_btn.description = "Show Trail Panel"
        trail_controls_box.layout.display = "none"
        trail_panel_container.layout.width = "180px"

toggle_trail_panel_btn.observe(_update_trail_panel_visibility, names=["value"])
_update_trail_panel_visibility()

trail_panel_control = WidgetControl(widget=trail_panel_container, position="bottomright")
m.add_control(trail_panel_control)

# Registry lives on the map instance for convenience
m._trail_layers = {}  # key -> dict(file_name, layer_group, ui_row)

def _extract_kml_bytes(file_name, raw_bytes):
    lower = (file_name or "").lower()
    if lower.endswith(".kmz"):
        with zipfile.ZipFile(io.BytesIO(raw_bytes), "r") as zf:
            kml_files = [n for n in zf.namelist() if n.lower().endswith(".kml")]
            if not kml_files:
                raise ValueError("KMZ contains no .kml file.")
            return zf.read(kml_files[0])
    if lower.endswith(".kml"):
        return raw_bytes
    raise ValueError("Unsupported file type.")

def _ns(root):
    nsmap = getattr(root, "nsmap", {}) or {}
    default_ns = nsmap.get(None, "http://www.opengis.net/kml/2.2")
    return {"k": default_ns}

def _parse_point_coords(text):
    first = text.strip().split()[0]
    parts = first.split(",")
    if len(parts) < 2:
        return None
    lon = float(parts[0])
    lat = float(parts[1])
    return (lat, lon)

def _parse_linestring_coords(text):
    pts = []
    for tup in text.strip().split():
        parts = tup.split(",")
        if len(parts) < 2:
            continue
        lon = float(parts[0])
        lat = float(parts[1])
        pts.append((lat, lon))
    return pts if len(pts) >= 2 else None

def _fit_bounds(latlons):
    lats = [p[0] for p in latlons]
    lons = [p[1] for p in latlons]
    if not lats:
        return
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])

def _parse_kml(kml_bytes):
    root = etree.fromstring(kml_bytes)
    ns = _ns(root)
    placemarks = root.xpath(".//k:Placemark", namespaces=ns) or root.xpath(".//Placemark")

    points = []
    lines = []

    for pm in placemarks:
        name = "Placemark"
        name_els = pm.xpath("./k:name", namespaces=ns) or pm.xpath("./name")
        if name_els and name_els[0].text:
            name = name_els[0].text.strip()

        pcoords = pm.xpath(".//k:Point/k:coordinates", namespaces=ns) or pm.xpath(".//Point/coordinates")
        if pcoords and pcoords[0].text:
            latlon = _parse_point_coords(pcoords[0].text)
            if latlon:
                points.append({"name": name, "lat": latlon[0], "lon": latlon[1]})

        lcoords = pm.xpath(".//k:LineString/k:coordinates", namespaces=ns) or pm.xpath(".//LineString/coordinates")
        if lcoords and lcoords[0].text:
            pts = _parse_linestring_coords(lcoords[0].text)
            if pts:
                lines.append({"name": name, "coords": pts})

    return points, lines

def _refresh_layers_box():
    layers_box.children = [rec["ui_row"] for rec in m._trail_layers.values()]

def _update_trail_status():
    n = len(m._trail_layers)
    if n == 0:
        trail_status.value = "<b>Trails:</b> none loaded"
    else:
        trail_status.value = f"<b>Trails:</b> {n} layer(s) loaded"

def _apply_color_to_layer(layer_group, color_hex):
    for lyr in getattr(layer_group, "layers", []):
        if isinstance(lyr, Polyline):
            lyr.color = color_hex
        elif isinstance(lyr, CircleMarker):
            lyr.color = color_hex
            lyr.fill_color = color_hex

def _set_layer_visibility(layer_group, visible: bool):
    for lyr in getattr(layer_group, "layers", []):
        lyr.visible = bool(visible)

def _remove_trail_layer(key):
    rec = m._trail_layers.get(key)
    if not rec:
        return
    lg = rec["layer_group"]
    try:
        m.remove_layer(lg)
    except Exception:
        pass
    del m._trail_layers[key]
    _refresh_layers_box()
    _update_trail_status()

def _add_trail_layer(file_name, raw_bytes, default_color="#FF0000"):
    kml_bytes = _extract_kml_bytes(file_name, raw_bytes)
    points, lines = _parse_kml(kml_bytes)
    if not points and not lines:
        raise ValueError("No supported Placemarks found (Points or LineStrings).")

    lg = LayerGroup()
    all_coords = []

    for ln in lines:
        coords = ln["coords"]
        all_coords.extend(coords)
        lg.add_layer(Polyline(locations=coords, weight=3, color=default_color))

    for p in points:
        lat, lon = p["lat"], p["lon"]
        all_coords.append((lat, lon))
        marker = CircleMarker(location=(lat, lon), radius=6, color=default_color, fill_color=default_color)
        marker.popup = Popup(child=widgets.HTML(value=f"<b>{p['name']}</b><br>{lat:.6f}, {lon:.6f}"))
        lg.add_layer(marker)

    m.add_layer(lg)

    label = widgets.HTML(value=f"{file_name}")
    color = widgets.ColorPicker(value=default_color, description="", layout=Layout(width="110px"))
    visible = widgets.Checkbox(value=True, description="Visible", indent=False, layout=Layout(width="90px"))
    remove_btn = widgets.Button(description="Remove", layout=Layout(width="70px"))

    key = f"{file_name}:{len(raw_bytes)}"

    def _on_color(change):
        _apply_color_to_layer(lg, change["new"])

    def _on_visible(change):
        _set_layer_visibility(lg, change["new"])

    def _on_remove(b=None):
        _remove_trail_layer(key)

    color.observe(_on_color, names=["value"])
    visible.observe(_on_visible, names=["value"])
    remove_btn.on_click(_on_remove)

    row = widgets.HBox([label, color, visible, remove_btn], layout=Layout(width="360px"))

    m._trail_layers[key] = {"file_name": file_name, "layer_group": lg, "ui_row": row}
    _refresh_layers_box()
    _update_trail_status()

    if all_coords:
        _fit_bounds(all_coords)

def _on_trail_upload_change(change):
    if not trail_upload.value:
        return

    payloads = []
    val = trail_upload.value

    if isinstance(val, dict):
        for name, meta in val.items():
            payloads.append((name, meta.get("content", None)))
    elif isinstance(val, (list, tuple)):
        for item in val:
            payloads.append((item.get("name", "uploaded"), item.get("content", None)))

    added = 0
    errors = []

    for file_name, raw_bytes in payloads:
        if raw_bytes is None:
            errors.append(f"{file_name}: missing content bytes")
            continue

        key = f"{file_name}:{len(raw_bytes)}"
        if key in m._trail_layers:
            continue

        try:
            defaults = ["#FF0000", "#00AA00", "#0066FF", "#FF9900", "#AA00AA", "#00AAAA"]
            default_color = defaults[len(m._trail_layers) % len(defaults)]
            _add_trail_layer(file_name, raw_bytes, default_color=default_color)
            added += 1
        except Exception as e:
            errors.append(f"{file_name}: {str(e)}")

    if added == 0 and not errors:
        trail_status.value = "<b>Trails:</b> no new files added (duplicates skipped)."
    elif errors:
        trail_status.value = "<b>Trails:</b> added some layers, with errors:<br>" + "<br>".join(errors)

def _on_trail_clear_clicked(b=None):
    for k in list(m._trail_layers.keys()):
        _remove_trail_layer(k)
    _update_trail_status()

trail_upload.observe(_on_trail_upload_change, names=["value"])
trail_clear_btn.on_click(_on_trail_clear_clicked)

# Auto-generate an initial flight path so the waypoint planner is visibly working
_regen_clicked()

m

## Next steps

- Add an exporter that emits DJI Fly compatible KMZ missions for Air 3.
- Optionally add a "Deploy" helper for copying the KMZ to the RC 2 via OpenMTP or ADB.
